In [1]:
#export MODEL="Your model here
export MODEL="unsloth/gemma-4-E4B-it-GGUF:UD-Q6_K_XL"

In [ ]:
llama-fit-params -hf $MODEL # This is just for downloading and some initial values to probe

In [2]:
# Configurable batch sizes to test to fit context
# Though it looks like batch sizes have no effects 
# as per README 
# "Increasing this value above the value of the physical batch size may improve prompt processing performance 
# when using multiple GPUs with pipeline parallelism. Default: `2048`" 

BATCH_SIZES="2048" # 2 args so we can see the diff, shoud not make a difference  
UBATCH_SIZES="64 128 256"
FITT="512"
CMOE="20 30 40 50"

echo "Testing different batch/ubatch/fitt/cmoe combinations for ${MODEL}..." 
  
for ub in $UBATCH_SIZES; do  
    for b in $BATCH_SIZES; do
        for ft in $FITT; do
            for cmoe in $CMOE; do
                # need to add fitt for nvidia
                OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub -fitt $ft -ncmoe $cmoe 2>&1)
                
                #echo "Raw output: $OUTPUT"  # Debug line  
                # Extract context and GPU layers more robustly  
                CTX=$(echo "$OUTPUT" | grep -o '\-c [0-9]*' | awk '{print $2}')  
                NGL=$(echo "$OUTPUT" | grep -o '\-ngl -\?[0-9]*' | awk '{print $2}')  

                echo -n "ub: $ub, b: $b, fitt: $ft, ncmoe: $cmoe"
                if [ ! -z "$CTX" ] && [ ! -z "$NGL" ]; then  
                    echo "ctx: $CTX, ngl: $NGL"  
                else  
                    echo "No valid parameters found"  
                fi
            done
        done
    done  
done 

ub: 256, b: 1024, fitt: 256, ctx: 4096, ngl: 42
ub: 256, b: 1024, fitt: 512, ctx: 4096, ngl: 40
ub: 256, b: 2048, fitt: 256, ctx: 4096, ngl: 42
ub: 256, b: 2048, fitt: 512, ctx: 4096, ngl: 40
ub: 512, b: 1024, fitt: 256, ctx: 4096, ngl: 40
ub: 512, b: 1024, fitt: 512, ctx: 4096, ngl: 37
ub: 512, b: 2048, fitt: 256, ctx: 4096, ngl: 40
ub: 512, b: 2048, fitt: 512, ctx: 4096, ngl: 37


In [ ]:
echo "Staring llama-bench for ${MODEL}..." 
llama-bench -hf $MODEL -ub 64 -ngl 0,20,99 -p 1000,2000 -n 256,512

In [ ]:
echo "Staring llama-cli for ${MODEL}..." 
llama-cli -lv 3 -hf $MODEL -ub 64 -fitt 256 -ngl -1 --no-warmup --no-mmproj --reasoning off --single-turn --prompt "Who are you?"

In [ ]:
echo "Staring llama-server for ${MODEL}..." 
llama-server -hf $MODEL --threads-http 1 -ub 64 -fitt 256 -ngl -1 --no-warmup --no-mmproj --reasoning off -np 1